In [1]:
import requests
from bs4 import BeautifulSoup

url = "https://alinino.az/"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
print(response.status_code)  # 200 olmalıdır
soup = BeautifulSoup(response.text, "html.parser")
print(soup.title.text)  # saytın adı çıxmalıdır

200
Alinino.az | Kitablar, Oyuncaqlar, Məktəb ləvazimatları və Hədiyyələr


In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

headers = {"User-Agent": "Mozilla/5.0"}
kitablar = []

for page in range(1, 8):  # saytda 7 səhifə var (Bestseller bölməsi)
    url = f"https://alinino.az/collection/bestseller-3?page={page}"
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    # Bütün kitab linklərini tapırıq (/product/ ilə başlayan)
    products = soup.find_all("a", href=lambda x: x and "/product/" in x)

    seen = set()
    for p in products:
        title = p.get_text(strip=True)
        link = p["href"]
        if not title or link in seen:
            continue
        seen.add(link)

        # Qiyməti tapmağa çalışaq (kartın "valideyn" konteynerində axtarırıq)
        parent = p.find_parent("div")
        qiymet_yeni, qiymet_kohne = None, None
        if parent:
            text = parent.get_text(" ", strip=True)
            import re
            qiymetler = re.findall(r"(\d+\.\d{2})\s*AZN", text)
            if qiymetler:
                qiymet_yeni = qiymetler[0]
                if len(qiymetler) > 1:
                    qiymet_kohne = qiymetler[1]

        kitablar.append({
            "Başlıq": title,
            "Yeni qiymət (AZN)": qiymet_yeni,
            "Köhnə qiymət (AZN)": qiymet_kohne,
            "Link": "https://alinino.az" + link if link.startswith("/") else link
        })

    print(f"Səhifə {page} işləndi, indiyə qədər {len(kitablar)} kitab")
    time.sleep(1)

df = pd.DataFrame(kitablar)
df = df.drop_duplicates(subset="Link")
df.to_csv("bestseller_kitablar.csv", index=False, encoding="utf-8-sig")
print(f"\nCƏMİ {len(df)} kitab yadda saxlanıldı")
df.head(10)

Səhifə 1 işləndi, indiyə qədər 87 kitab
Səhifə 2 işləndi, indiyə qədər 174 kitab
Səhifə 3 işləndi, indiyə qədər 261 kitab
Səhifə 4 işləndi, indiyə qədər 348 kitab
Səhifə 5 işləndi, indiyə qədər 435 kitab
Səhifə 6 işləndi, indiyə qədər 522 kitab
Səhifə 7 işləndi, indiyə qədər 579 kitab

CƏMİ 533 kitab yadda saxlanıldı


,Başlıq,Yeni qiymət (AZN),Köhnə qiymət (AZN),Link
0,Sürətli görünüş,None,None,https://alinino.az/product/min-mohtesem-gunes-3
1,0,None,None,https://alinino.az/product/min-mohtesem-gunes-...
2,Sürətli görünüş,None,None,https://alinino.az/product/sirli-bag-fd9d43
3,0,None,None,https://alinino.az/product/sirli-bag-fd9d43#re...
4,Sürətli görünüş,None,None,https://alinino.az/product/odisseya-5d4097
5,0,None,None,https://alinino.az/product/odisseya-5d4097#rev...
6,Sürətli görünüş,None,None,https://alinino.az/product/leyleklerin-ucusu
7,0,None,None,https://alinino.az/product/leyleklerin-ucusu#r...
8,Sürətli görünüş,None,None,https://alinino.az/product/mektubdaki-qadin-92...
9,0,None,None,https://alinino.az/product/mektubdaki-qadin-92...


In [5]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

headers = {"User-Agent": "Mozilla/5.0"}
kitablar = []

def qiymetleri_tap(tag):
    """Yuxarıya doğru gedib AZN qiymətini axtarır"""
    current = tag
    for _ in range(6):  # ən çox 6 səviyyə yuxarı
        current = current.find_parent()
        if current is None:
            break
        text = current.get_text(" ", strip=True)
        qiymetler = re.findall(r"(\d+\.\d{2})\s*AZN", text)
        if len(qiymetler) >= 1:
            return qiymetler
    return []

for page in range(1, 8):
    url = f"https://alinino.az/collection/bestseller-3?page={page}"
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    links = soup.find_all("a", href=lambda x: x and "/product/" in x and "#" not in x)

    seen = set()
    for link_tag in links:
        title = link_tag.get_text(strip=True)
        href = link_tag["href"]

        # Yalnız əsl başlıq linklərini saxla (boş, "Sürətli görünüş" olmayanları)
        if not title or title == "Sürətli görünüş" or title.isdigit():
            continue
        if href in seen:
            continue
        seen.add(href)

        qiymetler = qiymetleri_tap(link_tag)
        qiymet_yeni = qiymetler[0] if len(qiymetler) > 0 else None
        qiymet_kohne = qiymetler[1] if len(qiymetler) > 1 else None

        kitablar.append({
            "Başlıq": title,
            "Yeni qiymət (AZN)": qiymet_yeni,
            "Köhnə qiymət (AZN)": qiymet_kohne,
            "Link": "https://alinino.az" + href if href.startswith("/") else href
        })

    print(f"Səhifə {page} işləndi, indiyə qədər {len(kitablar)} kitab")
    time.sleep(1)

df = pd.DataFrame(kitablar)
df = df.drop_duplicates(subset="Link")
df.to_csv("bestseller_kitablar.csv", index=False, encoding="utf-8-sig")
print(f"\nCƏMİ {len(df)} kitab yadda saxlanıldı")
df.head(10)

Səhifə 1 işləndi, indiyə qədər 47 kitab
Səhifə 2 işləndi, indiyə qədər 94 kitab
Səhifə 3 işləndi, indiyə qədər 141 kitab
Səhifə 4 işləndi, indiyə qədər 188 kitab
Səhifə 5 işləndi, indiyə qədər 235 kitab
Səhifə 6 işləndi, indiyə qədər 282 kitab
Səhifə 7 işləndi, indiyə qədər 314 kitab

CƏMİ 268 kitab yadda saxlanıldı


,Başlıq,Yeni qiymət (AZN),Köhnə qiymət (AZN),Link
0,Min möhtəşəm günəş,9.59,11.99,https://alinino.az/product/min-mohtesem-gunes-3
1,Sirli bağ,8.49,9.99,https://alinino.az/product/sirli-bag-fd9d43
2,Odisseya,11.89,13.99,https://alinino.az/product/odisseya-5d4097
3,Leyləklərin uçuşu,15.26,17.95,https://alinino.az/product/leyleklerin-ucusu
4,Məktubdakı qadın,11.01,12.95,https://alinino.az/product/mektubdaki-qadin-92...
5,Mirvari sırğalı qız,11.86,13.95,https://alinino.az/product/mirvari-sirgali-qiz
6,Zəfər şəhəri,16.11,18.95,https://alinino.az/product/zefer-seheri
7,Yevgeni Onegin,8.49,9.99,https://alinino.az/product/yevgeni-onegin-2
8,Gecə musiqisi,12.71,14.95,https://alinino.az/product/gece-musiqisi
9,Atuan türbələri,9.31,10.95,https://alinino.az/product/atuan-turbeleri


In [7]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

headers = {"User-Agent": "Mozilla/5.0"}

def qiymetleri_tap(tag):
    current = tag
    for _ in range(6):
        current = current.find_parent()
        if current is None:
            break
        text = current.get_text(" ", strip=True)
        qiymetler = re.findall(r"(\d+\.\d{2})\s*AZN", text)
        if len(qiymetler) >= 1:
            return qiymetler
    return []

kateqoriyalar = {
    "Bestseller": "bestseller-3",
    "Detektiv. Triller": "detektivy-trillery-2",
    "Fantastika": "fantastika-uzhasy",
    "Sevgi Romanı": "lyubovnye-romany-2",
    "Dünya Klassikası": "mirovaya-klassika",
    "Poeziya": "poeziya-2",
}

butun_kitablar = []

for janr_adi, slug in kateqoriyalar.items():
    page = 1
    while True:
        url = f"https://alinino.az/collection/{slug}?page={page}"
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.text, "html.parser")

        links = soup.find_all("a", href=lambda x: x and "/product/" in x and "#" not in x)
        if not links:
            break

        seen_this_page = set()
        yeni_elave = 0
        for link_tag in links:
            title = link_tag.get_text(strip=True)
            href = link_tag["href"]
            if not title or title == "Sürətli görünüş" or title.isdigit():
                continue
            if href in seen_this_page:
                continue
            seen_this_page.add(href)

            qiymetler = qiymetleri_tap(link_tag)
            qiymet_yeni = qiymetler[0] if len(qiymetler) > 0 else None
            qiymet_kohne = qiymetler[1] if len(qiymetler) > 1 else None

            butun_kitablar.append({
                "Janr": janr_adi,
                "Başlıq": title,
                "Yeni qiymət (AZN)": qiymet_yeni,
                "Köhnə qiymət (AZN)": qiymet_kohne,
                "Link": "https://alinino.az" + href if href.startswith("/") else href
            })
            yeni_elave += 1

        print(f"{janr_adi} - səhifə {page}: {yeni_elave} kitab")

        if page >= 7:
            break
        page += 1
        time.sleep(1)

df = pd.DataFrame(butun_kitablar)
df = df.drop_duplicates(subset="Link")
df.to_csv("janrli_kitablar.csv", index=False, encoding="utf-8-sig")
print(f"\nCƏMİ {len(df)} kitab yadda saxlanıldı (janr sütunu ilə)")
df.head(10)

Bestseller - səhifə 1: 47 kitab
Bestseller - səhifə 2: 47 kitab
Bestseller - səhifə 3: 47 kitab
Bestseller - səhifə 4: 47 kitab
Bestseller - səhifə 5: 47 kitab
Bestseller - səhifə 6: 47 kitab
Bestseller - səhifə 7: 32 kitab
Detektiv. Triller - səhifə 1: 56 kitab
Detektiv. Triller - səhifə 2: 56 kitab
Detektiv. Triller - səhifə 3: 56 kitab
Detektiv. Triller - səhifə 4: 56 kitab
Detektiv. Triller - səhifə 5: 56 kitab
Detektiv. Triller - səhifə 6: 56 kitab
Detektiv. Triller - səhifə 7: 56 kitab
Fantastika - səhifə 1: 56 kitab
Fantastika - səhifə 2: 56 kitab
Fantastika - səhifə 3: 56 kitab
Fantastika - səhifə 4: 56 kitab
Fantastika - səhifə 5: 56 kitab
Fantastika - səhifə 6: 56 kitab
Fantastika - səhifə 7: 56 kitab
Sevgi Romanı - səhifə 1: 49 kitab
Sevgi Romanı - səhifə 2: 49 kitab
Sevgi Romanı - səhifə 3: 49 kitab
Sevgi Romanı - səhifə 4: 29 kitab
Sevgi Romanı - səhifə 5: 9 kitab
Sevgi Romanı - səhifə 6: 9 kitab
Sevgi Romanı - səhifə 7: 9 kitab
Dünya Klassikası - səhifə 1: 52 kitab
Dünya 

,Janr,Başlıq,Yeni qiymət (AZN),Köhnə qiymət (AZN),Link
0,Bestseller,Min möhtəşəm günəş,9.59,11.99,https://alinino.az/product/min-mohtesem-gunes-3
1,Bestseller,Sirli bağ,8.49,9.99,https://alinino.az/product/sirli-bag-fd9d43
2,Bestseller,Odisseya,11.89,13.99,https://alinino.az/product/odisseya-5d4097
3,Bestseller,Leyləklərin uçuşu,15.26,17.95,https://alinino.az/product/leyleklerin-ucusu
4,Bestseller,Məktubdakı qadın,11.01,12.95,https://alinino.az/product/mektubdaki-qadin-92...
5,Bestseller,Mirvari sırğalı qız,11.86,13.95,https://alinino.az/product/mirvari-sirgali-qiz
6,Bestseller,Zəfər şəhəri,16.11,18.95,https://alinino.az/product/zefer-seheri
7,Bestseller,Yevgeni Onegin,8.49,9.99,https://alinino.az/product/yevgeni-onegin-2
8,Bestseller,Gecə musiqisi,12.71,14.95,https://alinino.az/product/gece-musiqisi
9,Bestseller,Atuan türbələri,9.31,10.95,https://alinino.az/product/atuan-turbeleri
